# Task 3: RAG Pipeline: Qualitative Evaluation

**Branch:** `feat/full-vector-store`

Task 3's brief asks for a small, human-reviewed evaluation: "Create a
list of 5-10 representative questions... run your RAG pipeline and
analyze the results... Create an evaluation table with columns:
Question, Generated Answer, Retrieved Sources (1-2), Quality Score
(1-5), and Comments/Analysis."

This notebook is that deliverable 8 hand-picked, representative
questions (2 per product category), run through the real
`retrieve()` + `generate_answer()` pipeline (same code path `app.py`
uses, not a reimplementation).

**How this relates to `src/evaluation.py`:** that module is the
*systematic* follow-up 18 questions, LLM-as-judge scoring, results
tracked over time in `eval_results/`. This notebook is the smaller,
qualitative deliverable Task 3 explicitly asks for; `src/evaluation.py`
is what the original submission's grader feedback flagged as missing
("evaluation currently lives in a qualitative notebook with no fixed
question set, no scoring rubric, no repeatable metric tracking"). Both
exist here, this notebook isn't superseded by the harness, it's what
the harness was built *in addition to*.

**Note on the quality scores below:** `src/evaluation.py`'s
`judge_answer()` is reused here to generate a *suggested* score and
comment per question that saves re-deriving a judge prompt, but an
LLM's self-assessment of a RAG system's own output is not a substitute
for actually reading the retrieved sources and the generated answer
yourself. Review and edit every score/comment below before pasting this
table into the report.


In [2]:
import sys
sys.path.insert(0, "..")

from src.embedding import load_embedding_model, EMBEDDING_MODEL_NAME
from src.retriever import load_vector_store, build_retriever, RetrievalError
from src.generator import build_llm_client, generate_answer
from src.evaluation import judge_answer, to_markdown_table

# Defaults to the full pre-built store (Tasks 3-4's actual target, per
# the brief: "load the pre-built vector store... covers the complete
# filtered dataset"). Swap to vector_store_sample/complaints_sample for
# a much faster local iteration loop while drafting this notebook, then
# switch back before generating the final table for the report.
VECTOR_STORE_DIR = "../vector_store_full"
COLLECTION_NAME = "complaints_full"


In [3]:
embed_model = load_embedding_model(EMBEDDING_MODEL_NAME)
collection = load_vector_store(VECTOR_STORE_DIR, COLLECTION_NAME)
retrieve = build_retriever(collection, model=embed_model)

llm_client = build_llm_client()
judge_client = build_llm_client()  # same model is fine for a one-off qualitative pass


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1639.28it/s]


Loaded collection 'complaints_full' with 1,375,327 chunks


## The 8 representative questions

Two per product category, chosen to cover distinct complaint themes
within each category rather than near-duplicates of each other (e.g.
credit card *fraud* vs. credit card *billing disputes*, not two
phrasings of the same question).


In [4]:
QUESTIONS = [
    {"id": "q1", "category": "Credit Card", "question": "Why are customers unhappy with their credit cards?"},
    {"id": "q2", "category": "Credit Card", "question": "What fraud or unauthorized-charge issues are credit card customers reporting?"},
    {"id": "q3", "category": "Personal Loan", "question": "What are the main complaints about personal loans?"},
    {"id": "q4", "category": "Personal Loan", "question": "Do customers report unclear or misleading personal loan terms?"},
    {"id": "q5", "category": "Savings Account", "question": "What problems are customers having with savings accounts?"},
    {"id": "q6", "category": "Savings Account", "question": "Are customers complaining about unexpected fees on their savings accounts?"},
    {"id": "q7", "category": "Money Transfer", "question": "What issues are customers facing with money transfers?"},
    {"id": "q8", "category": "Money Transfer", "question": "Are customers reporting delayed or failed money transfers?"},
]
print(f"{len(QUESTIONS)} questions across {len(set(q['category'] for q in QUESTIONS))} categories")


8 questions across 4 categories


## Run the pipeline and review each result

In [5]:
results = []

for item in QUESTIONS:
    qid, category, question = item["id"], item["category"], item["question"]
    print(f"=== [{qid}] {question} ===")

    try:
        chunks = retrieve(question, top_k=5)
    except RetrievalError as exc:
        print(f"  RETRIEVAL FAILED: {exc}\n")
        results.append({
            "id": qid, "category": category, "question": question,
            "answer": None, "retrieved_sources": [],
            "retrieval_relevance": None, "faithfulness": None,
            "comment": f"Retrieval failed: {exc}", "judge_error": str(exc),
        })
        continue

    gen_result = generate_answer(llm_client, question, chunks)
    judge = judge_answer(judge_client, question, gen_result["context_block"], gen_result["answer"])

    print(f"  Answer: {gen_result['answer'][:300]}")
    print(f"  Top sources:")
    for c in chunks[:2]:
        print(f"    [{c['product_category']}] #{c['complaint_id']} "
              f"(similarity={c['similarity_score']:.3f}, company={c['company'] or 'n/a'})")
        print(f"      {c['chunk_text'][:150]}...")
    print(f"  Suggested score -- relevance: {judge['retrieval_relevance']}, "
          f"faithfulness: {judge['faithfulness']}")
    print(f"  Suggested comment: {judge['comment']}\n")

    results.append({
        "id": qid,
        "category": category,
        "question": question,
        "answer": gen_result["answer"],
        "retrieved_sources": [
            {"complaint_id": c["complaint_id"], "product_category": c["product_category"],
             "similarity_score": c["similarity_score"], "excerpt": c["chunk_text"][:200]}
            for c in chunks[:2]
        ],
        "generation_error": gen_result["error"],
        **judge,
    })


=== [q1] Why are customers unhappy with their credit cards? ===
  Answer: Customers are unhappy with their credit cards due to several reasons:
- Credit limits being reduced without justification, which increases their credit usage.
- Poor customer service and misinformation from customer service representatives.
- Unexpected changes such as increased interest rates, whic
  Top sources:
    [Credit Card] #8892605 (similarity=0.435, company=DISCOVER BANK)
      creditors. i have an exceptional payment history. there was no reason for them to reduce my credit limit at all doing so caused me harm by making my c...
    [Credit Card] #3856544 (similarity=0.424, company=CITIBANK, N.A.)
      h customers during these difficult time??? that is not our fault? credit card companies need to do what the government asked them to do help consumers...
  Suggested score -- relevance: 5, faithfulness: 5
  Suggested comment: Both dimensions are well-supported by the provided excerpts.

=== [q2] What fra

LLM call failed on attempt 1/3 (non-retryable or exhausted): Client error '402 Payment Required' for url 'https://router.huggingface.co/v1/chat/completions' (Request ID: Root=1-6a6b46ef-28e4f7a47ed4467673059164;52d970b2-1839-4af4-bbd7-0ab66fbbd6db)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.
LLM generation failed after retries: Client error '402 Payment Required' for url 'https://router.huggingface.co/v1/chat/completions' (Request ID: Root=1-6a6b46ef-28e4f7a47ed4467673059164;52d970b2-1839-4af4-bbd7-0ab66fbbd6db)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more includ

  Answer: I'm having trouble reaching the answer-generation service right now. Please try again in a moment. If this keeps happening, the retrieved sources below are still a good starting point.
  Top sources:
    [Savings Account] #3791508 (similarity=0.461, company=BANK OF AMERICA, NATIONAL ASSOCIATION)
      cluded on a small line on one of my statements ''. this should have been sent out as a big bank notification, like they send me an email every time my...
    [Savings Account] #11897139 (similarity=0.406, company=TD BANK US HOLDING COMPANY)
      formal complaint regarding excessive fees on my account dear td bank customer service, i am writing to formally dispute the excessive fees that have b...
  Suggested score -- relevance: None, faithfulness: None
  Suggested comment: 

=== [q7] What issues are customers facing with money transfers? ===


LLM call failed on attempt 1/3 (non-retryable or exhausted): Client error '402 Payment Required' for url 'https://router.huggingface.co/v1/chat/completions' (Request ID: Root=1-6a6b46f0-3f77100a3c233e692f073345;721e4550-3f5f-4deb-963d-4a34c8b25aa3)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.
LLM generation failed after retries: Client error '402 Payment Required' for url 'https://router.huggingface.co/v1/chat/completions' (Request ID: Root=1-6a6b46f0-3f77100a3c233e692f073345;721e4550-3f5f-4deb-963d-4a34c8b25aa3)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more includ

  Answer: I'm having trouble reaching the answer-generation service right now. Please try again in a moment. If this keeps happening, the retrieved sources below are still a good starting point.
  Top sources:
    [Money Transfer] #2802423 (similarity=0.433, company=Early Warning Services, LLC)
      les and money transfers rather than passing the entire burden to the customers to do that. moreover they negatively impacted all the customers. this i...
    [Money Transfer] #11520794 (similarity=0.423, company=JPMORGAN CHASE & CO.)
      problems with money transfer, receiving and sending out....
  Suggested score -- relevance: None, faithfulness: None
  Suggested comment: 

=== [q8] Are customers reporting delayed or failed money transfers? ===


LLM call failed on attempt 1/3 (non-retryable or exhausted): Client error '402 Payment Required' for url 'https://router.huggingface.co/v1/chat/completions' (Request ID: Root=1-6a6b46f0-195b69937622391644564f30;e62719ae-9081-4e75-b42c-fc0517327a1f)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.
LLM generation failed after retries: Client error '402 Payment Required' for url 'https://router.huggingface.co/v1/chat/completions' (Request ID: Root=1-6a6b46f0-195b69937622391644564f30;e62719ae-9081-4e75-b42c-fc0517327a1f)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more includ

  Answer: I'm having trouble reaching the answer-generation service right now. Please try again in a moment. If this keeps happening, the retrieved sources below are still a good starting point.
  Top sources:
    [Money Transfer] #4859023 (similarity=0.348, company=XOOM CORPORATION)
      good day, my name is , new jersey, . i am complaing xoom money transfer company a company , , , ca , phone for delaying these 2 specific transactions ...
    [Credit Card] #8252398 (similarity=0.322, company=Experian Information Solutions Inc.)
      ensuring prompt payments has always been my practice on this account, and i have never experienced delays. i am unsure about the reasons behind report...
  Suggested score -- relevance: None, faithfulness: None
  Suggested comment: 



## Evaluation table (for the report)

Reuses `src/evaluation.py`'s `to_markdown_table()` for the exact column
format the brief asks for: Question, Generated Answer, Retrieved
Sources, Quality Score, Comments/Analysis.

**Before pasting this into the report:** read back through the printed
output above and edit any score/comment that doesn't match your own
read of the answer and sources -- especially any row where the answer
looks off-topic, unsupported by the sources shown, or where "Quality
Score" doesn't reflect how you'd actually rate it.


In [6]:
table_input = {"rows": results}
print(to_markdown_table(table_input))


| Question | Generated Answer | Retrieved Sources | Quality Score | Comments/Analysis |
|---|---|---|---|---|
| Why are customers unhappy with their credit cards? | Customers are unhappy with their credit cards due to several reasons: - Credit limits being reduced without justification, which increases their credi... | Credit Card #8892605 (0.43); Credit Card #3856544 (0.42) | R:5/F:5 | Both dimensions are well-supported by the provided excerpts. |
| What fraud or unauthorized-charge issues are credit card customers reporting? | Credit card customers are reporting issues with fraudulent charges on their cards. These issues include: - Multiple cards (debit and credit) being sto... | Credit Card #5427299 (0.54); Credit Card #1941325 (0.51) | R:5/F:5 | The answer is fully based on the provided excerpts and accurately reflects the issues reported. |
| What are the main complaints about personal loans? | The main complaints about personal loans include: - Deceptive and misleading practices 

## What to look for while reviewing

- **Faithfulness**: does the answer only state things present in the
  retrieved excerpts, or does it add plausible-sounding claims the
  sources don't actually support?
- **Relevance**: are the top-2 sources shown actually about the
  question asked, or just the least-bad match available (a low
  similarity score, e.g. well under 0.2, is a signal worth checking)?
- **Category leakage**: does a Credit Card question ever pull a Savings
  Account source, or similar? A little cross-category retrieval isn't
  automatically wrong (some complaints genuinely span topics), but
  worth a second look if it happens often.
- **"I don't have enough information"**: the prompt template
  instructs the model to say this rather than guess -- if it never
  appears across all 8 questions, that's not necessarily a good sign;
  it may mean the model is filling gaps instead of admitting them.

## Next step

This qualitative table satisfies Task 3's deliverable. For the
systematic, quantitative pass across all 4 categories (18 questions,
tracked over time, informing the "85% faithfulness" success metric),
run:

```bash
python -m src.evaluation --vector-store ./vector_store_full --collection complaints_full
```
